In [ ]:
#Adrian Felipe Romero Barajas 01251152018
#Nikolas Peña Vergugo 01251152021
#Antonio Jose Rincon Herrera 01251152001
import csv
import os

# Detectar automáticamente si estamos en Google Colab
try:
    import google.colab
    enColab = True
except ImportError:
    enColab = False

# Base de datos de Departamentos y Municipios para la selección
departamentosMunicipios = {
    "Santander": ["Bucaramanga", "Floridablanca", "Girón", "Piedecuesta", "Barrancabermeja"],
    "Antioquia": ["Medellín", "Envigado", "Itagüí", "Bello", "Rionegro"],
    "Cundinamarca": ["Bogotá", "Soacha", "Chía", "Zipaquirá", "Facatativá"],
    "Valle del Cauca": ["Cali", "Palmira", "Buenaventura", "Tuluá", "Cartago"],
    "Atlántico": ["Barranquilla", "Soledad", "Malambo", "Sabanalarga", "Puerto Colombia"]
}

# Lista para guardar los municipios registrados
municipios = []
archivoCsv = "municipios.csv"

def cargarDesdeCsv():
    """Carga los municipios desde un archivo CSV predeterminado si existe."""
    if os.path.exists(archivoCsv):
        try:
            with open(archivoCsv, mode='r', encoding='utf-8') as file:
                reader = csv.reader(file)
                next(reader, None)  # Saltar la cabecera
                for row in reader:
                    if row:
                        municipio = row[0]
                        urbano = int(row[1])
                        rural = int(row[2])
                        total = int(row[3])
                        porRural = float(row[4])
                        porUrbano = float(row[5])
                        categoria = row[6]
                        municipios.append([municipio, urbano, rural, total, porRural, porUrbano, categoria])
            print(f"¡Datos cargados exitosamente desde '{archivoCsv}'!")
        except Exception as e:
            print(f"Error al cargar el archivo CSV: {e}")

def guardarEnCsvAutomatico():
    """Guarda automáticamente en el archivo local de Colab."""
    try:
        with open(archivoCsv, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["municipios", "Poblacion_Urbana", "Poblacion_Rural", "Poblacion_Total", "Porcentaje_Rural", "Porcentaje_Urbano", "Categoria"])
            for m in municipios:
                writer.writerow(m)
    except Exception as e:
        print(f"Error al guardar automáticamente: {e}")

def seleccionarDepartamentoYMunicipio():
    """Permite seleccionar un departamento y luego un municipio mediante listas numeradas."""
    print("\n--- SELECCIÓN DE DEPARTAMENTO ---")
    listaDepartamentos = list(departamentosMunicipios.keys())
    for idx, dep in enumerate(listaDepartamentos, 1):
        print(f"{idx}. {dep}")

    try:
        depIdx = int(input("Seleccione un departamento (número): ")) - 1
        if depIdx < 0 or depIdx >= len(listaDepartamentos):
            print("Opción fuera de rango.")
            return None
        departamentoSeleccionado = listaDepartamentos[depIdx]
    except ValueError:
        print("Entrada no válida.")
        return None

    print(f"\n--- MUNICIPIOS DE {departamentoSeleccionado.upper()} ---")
    listaMuni = departamentosMunicipios[departamentoSeleccionado]
    for idx, mun in enumerate(listaMuni, 1):
        print(f"{idx}. {mun}")

    try:
        munIdx = int(input("Seleccione un municipio (número): ")) - 1
        if munIdx < 0 or munIdx >= len(listaMuni):
            print("Opción fuera de rango.")
            return None
        return listaMuni[munIdx]
    except ValueError:
        print("Entrada no válida.")
        return None

# Cargar datos al iniciar
cargarDesdeCsv()

opcion = 0

while opcion != 7:
    print("\n-MENÚ PRINCIPAL-")
    print("1. Agregar municipio")
    print("2. Editar datos de un municipio")
    print("3. Eliminar municipio")
    print("4. Mostrar todos los municipios")
    print("5. Importar datos (Subir CSV desde tu PC)")
    print("6. Descargar / Guardar CSV a tu PC")
    print("7. Salir")

    try:
        opcion = int(input("Ingrese una opción (1-7): "))
    except ValueError:
        print("Por favor, ingrese un número válido.")
        continue

    # 1. Agregar
    if opcion == 1:
        try:
            cantidadDeMunicipios = int(input("¿Cuántos municipios desea ingresar?: "))
        except ValueError:
            print("Cantidad no válida.")
            continue

        for i in range(cantidadDeMunicipios):
            print(f"\n--- Municipio #{i + 1} ---")
            nombreMunicipio = seleccionarDepartamentoYMunicipio()

            if not nombreMunicipio:
                print("No se pudo seleccionar el municipio. Intente de nuevo.")
                continue

            try:
                personasCascoUrbano = int(input(f"Ingresa la cantidad de personas del casco urbano para {nombreMunicipio}: "))
                personasCascoRural = int(input(f"Ingresa la cantidad de personas del sector rural para {nombreMunicipio}: "))
            except ValueError:
                print("Debe ingresar números enteros válidos para la población.")
                continue

            totalPersonasMunicipio = personasCascoUrbano + personasCascoRural
            if totalPersonasMunicipio == 0:
                print("La población total no puede ser cero.")
                continue

            porcentage1 = (personasCascoRural * 100) / totalPersonasMunicipio
            porcentage2 = (personasCascoUrbano * 100) / totalPersonasMunicipio

            if porcentage2 < 40:
                categoria = "rural"
            elif porcentage2 < 60:
                categoria = "semiurbano"
            else:
                categoria = "urbana"

            municipios.append([nombreMunicipio, personasCascoUrbano, personasCascoRural, totalPersonasMunicipio, porcentage1, porcentage2, categoria])

            print(f"\nLa cantidad de personas del municipio de {nombreMunicipio} son: {totalPersonasMunicipio} personas")
            print(f"{nombreMunicipio} tiene un {round(porcentage1, 2)}% de personas del casco rural y un {round(porcentage2, 2)}% de personas del casco urbano")
            print(f"La categoría del municipio es: {categoria}")

        guardarEnCsvAutomatico()

    # 2. Editar
    elif opcion == 2:
        if len(municipios) == 0:
            print("No hay municipios registrados para editar.")
        else:
            nombreBuscar = str(input("Ingrese el nombre del municipio que desea editar: "))
            encontrado = False

            for i in range(len(municipios)):
                if municipios[i][0].lower() == nombreBuscar.lower():
                    print(f"Editando municipio: {municipios[i][0]}")
                    personasCascoUrbano = int(input("Ingresa la nueva cantidad de personas del casco urbano: "))
                    personasCascoRural = int(input("Ingresa la nueva cantidad de personas del sector rural: "))

                    totalPersonasMunicipio = personasCascoUrbano + personasCascoRural
                    porcentage1 = (personasCascoRural * 100) / totalPersonasMunicipio
                    porcentage2 = (personasCascoUrbano * 100) / totalPersonasMunicipio

                    if porcentage2 < 40:
                        categoria = "rural"
                    elif porcentage2 < 60:
                        categoria = "semiurbano"
                    else:
                        categoria = "urbana"

                    municipios[i] = [municipios[i][0], personasCascoUrbano, personasCascoRural, totalPersonasMunicipio, porcentage1, porcentage2, categoria]
                    print(f"Datos de {municipios[i][0]} actualizados correctamente!")
                    encontrado = True
                    break

            if not encontrado:
                print("El municipio ingresado no se encuentra en la lista.")
            else:
                guardarEnCsvAutomatico()

    # 3. Eliminar
    elif opcion == 3:
        if len(municipios) == 0:
            print("No hay municipios registrados para eliminar.")
        else:
            nombreBuscar = str(input("Ingrese el nombre del municipio que desea eliminar: "))
            encontrado = False

            for i in range(len(municipios)):
                if municipios[i][0].lower() == nombreBuscar.lower():
                    municipioEliminado = municipios.pop(i)
                    print(f"El municipio '{municipioEliminado[0]}' ha sido eliminado correctamente.")
                    encontrado = True
                    break

            if not encontrado:
                print("El municipio ingresado no se encuentra en la lista.")
            else:
                guardarEnCsvAutomatico()

    # 4. Mostrar
    elif opcion == 4:
        if len(municipios) == 0:
            print("No hay municipios registrados.")
        else:
            print("-LISTA DE MUNICIPIOS-")
            for i in range(len(municipios)):
                m = municipios[i]
                print(f" - Municipio: {m[0]}")
                print(f"   - Población urbana: {m[1]} ({round(m[5], 2)}%)")
                print(f"   - Población rural: {m[2]} ({round(m[4], 2)}%)")
                print(f"   - Total población: {m[3]}")
                print(f"   - Categoría: {m[6]}")

    # 5. Importar datos (Subir CSV)
    elif opcion == 5:
        if enColab:
            print("Aparecerá un botón abajo para elegir y subir tu archivo CSV:")
            from google.colab import files
            uploaded = files.upload()
            for filename in uploaded.keys():
                try:
                    with open(filename, mode='r', encoding='utf-8') as file:
                        reader = csv.reader(file)
                        next(reader, None)
                        count = 0
                        for row in reader:
                            if row and len(row) >= 7:
                                municipio = row[0]
                                urbano = int(row[1])
                                rural = int(row[2])
                                total = int(row[3])
                                porRural = float(row[4])
                                porUrbano = float(row[5])
                                categoria = row[6]
                                municipios.append([municipio, urbano, rural, total, porRural, porUrbano, categoria])
                                count += 1
                    print(f"¡Se han importado {count} municipios exitosamente desde '{filename}'!")
                    guardarEnCsvAutomatico()
                except Exception as e:
                    print(f"Error al procesar el archivo: {e}")
        else:
            print("Esta opción está diseñada para Google Colab.")

    # 6. Descargar / Guardar CSV
    elif opcion == 6:
        if len(municipios) == 0:
            print("No hay datos para exportar/descargar.")
        else:
            guardarEnCsvAutomatico()
            if enColab:
                print("Iniciando descarga del archivo CSV...")
                from google.colab import files
                files.download(archivoCsv)
            else:
                print(f"Los datos se guardaron en el archivo local: {archivoCsv}")

    # 7. Salir
    elif opcion == 7:
        guardarEnCsvAutomatico()
        print("¡Gracias por usar el sistema! Hasta luego.")

    else:
        opcion = 0
        print("Opción no válida. Por favor ingrese un número del 1 al 7.")

¡Datos cargados exitosamente desde 'municipios.csv'!

-MENÚ PRINCIPAL-
1. Agregar municipio
2. Editar datos de un municipio
3. Eliminar municipio
4. Mostrar todos los municipios
5. Importar datos (Subir CSV desde tu PC)
6. Descargar / Guardar CSV a tu PC
7. Salir
Ingrese una opción (1-7): 1
¿Cuántos municipios desea ingresar?: 1

--- Municipio #1 ---

--- SELECCIÓN DE DEPARTAMENTO ---
1. Santander
2. Antioquia
3. Cundinamarca
4. Valle del Cauca
5. Atlántico
